[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maercaestro/megat/blob/main/notebooks/05_synthetic_story_generation.ipynb)

# Notebook 05 — Synthetic Malay Fiction Generation + LLM Judge

Generates Bahasa Melayu short stories using GPT-4o-mini, scores each one with an LLM judge, filters by quality, and saves to `maercaestro/cereka-kecil` on HuggingFace.

**Model:** `gpt-4o-mini` (10M free tokens/day)  
**Output dataset:** `maercaestro/cereka-kecil`  
**Token budget per story:** ~2,000 tokens (generation + judging)  
**Stories per day:** ~5,000 (within 10M token limit)

---
**Style anchors from `abuworks`:**
- Colloquial BM, short burst sentences, direct reader address
- Malaysian institutional settings (kilang, asrama, kampung, pejabat, hospital)
- Genre subversion — starts as one thing, pivots to another
- Supernatural kept ambiguous and unnamed
- Islamic protective practices as story beats (Yassin, ustaz, 3AM, malam Jumaat)
- Physical sensation over explicit description

## Step 1 — Install dependencies

In [ ]:
!pip install -q openai huggingface_hub datasets

## Step 2 — Configuration

In [ ]:
import os
from google.colab import userdata

# ── API keys ────────────────────────────────────────────────────────
# Store these in Colab Secrets (key icon on left sidebar)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
HF_TOKEN       = userdata.get('HF_TOKEN')

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# ── Model ───────────────────────────────────────────────────────────
GEN_MODEL    = 'gpt-4o-mini'   # story generation
JUDGE_MODEL  = 'gpt-4o-mini'   # LLM judge (same model, different prompt)

# ── Dataset ─────────────────────────────────────────────────────────
HF_REPO_ID   = 'maercaestro/cereka-kecil'

# ── Generation settings ─────────────────────────────────────────────
TARGET_STORIES    = 500      # total stories to generate this session
PUSH_EVERY        = 50       # push to HF every N accepted stories
QUALITY_THRESHOLD = 7.0      # minimum average judge score to keep (out of 10)
CHECKPOINT_FILE   = 'cereka_checkpoint.json'

print(f'Model:      {GEN_MODEL}')
print(f'Target:     {TARGET_STORIES} stories')
print(f'Threshold:  {QUALITY_THRESHOLD}/10')
print(f'Dataset:    {HF_REPO_ID}')

## Step 3 — Genre and prompt matrix

Designed from the style analysis of `abuworks`. Primary genres are horror and sci-fi, expanded with diverse supporting genres. Each combination produces a distinct story type.

In [ ]:
import random

# ── Genre weights ───────────────────────────────────────────────────
# Higher weight = more stories from that genre
GENRES = [
    ('Seram / Supranatural',    0.25),   # primary — horror/supernatural
    ('Sains Fiksyen',           0.15),   # primary — sci-fi
    ('Misteri / Thriller',      0.12),   # mystery/thriller
    ('Seram Psikologi',         0.10),   # psychological horror
    ('Dystopia / Cyberpunk',    0.08),   # speculative
    ('Sejarah Seram',           0.08),   # historical horror
    ('Drama Keluarga Seram',    0.07),   # family drama with supernatural twist
    ('Komedi Hitam',            0.06),   # dark comedy (like Zon Larangan)
    ('Thriller Pejabat',        0.05),   # workplace thriller
    ('Lagenda / Folklore',      0.04),   # Malay folklore
]

SETTINGS = [
    'kilang minyak atau loji penapisan',
    'sekolah berasrama penuh semasa cuti sekolah',
    'kampung terpencil dengan tanah perkuburan lama',
    'hospital kerajaan lewat malam',
    'bangunan pejabat korporat KL',
    'hutan belantara atau kawasan pergunungan',
    'dunia maya atau realiti maya',
    'pekan kecil yang sedang mati',
    'flat kos rendah di bandar',
    'stesen LRT atau MRT tengah malam',
    'kampus universiti semasa minggu peperiksaan',
    'kapal atau bot di lautan',
    'Malaysia zaman penjajahan British',
    'bandar pintar futuristik Malaysia 2080',
    'dusun atau kebun getah',
]

TONES = [
    'slow-burn seram yang membina ketegangan perlahan',
    'aksi pantas dengan twist mengejut di akhir',
    'melankoli dan nostalgia yang bertukar menjadi gelap',
    'suara narator santai dan jenaka yang tiba-tiba serius',
    'teknikal dan profesional yang diserap oleh yang ganjil',
    'autobiografi — narator bercerita terus kepada pembaca',
    'dread kosmik — ancaman yang terlalu besar untuk difahami',
    'tension psikologi — watak tidak pasti mana yang nyata',
]

PERSPECTIVES = [
    'orang pertama (aku/saya) — narator dalam cerita',
    'orang ketiga rapat — mengikut satu watak sahaja',
]

SUPERNATURAL_ELEMENTS = [
    'bunyi yang tidak dapat dikenal pasti sumbernya',
    'rekod kamera CCTV yang menunjukkan sesuatu yang tidak sepatutnya ada',
    'entiti dalam jubah atau baju putih yang menonton dari jauh',
    'gangguan mekanikal atau elektrik yang berulang pada masa yang sama',
    'mimpi yang terlalu nyata dan meninggalkan kesan fizikal',
    'orang yang menghilang tanpa jejak dalam kawasan tertutup',
    'suara atau bisikan dalam rakaman audio',
    'watak yang berkelakuan berbeza selepas mengunjungi lokasi tertentu',
    'tidak ada elemen luar biasa — hanya suasana dan ketidakpastian sahaja',
]

TRIGGERS = [
    'berlaku pada malam Jumaat',
    'berlaku tepat jam 3 pagi',
    'berlaku semasa watak bersendirian buat kali pertama',
    'dipicu oleh watak luar memasuki ruang yang bukan miliknya',
    'bermula dengan kejadian kecil yang diabaikan',
]

def sample_prompt_config():
    """Sample a random combination of story parameters."""
    genre   = random.choices([g[0] for g in GENRES], weights=[g[1] for g in GENRES])[0]
    setting = random.choice(SETTINGS)
    tone    = random.choice(TONES)
    pov     = random.choice(PERSPECTIVES)
    element = random.choice(SUPERNATURAL_ELEMENTS)
    trigger = random.choice(TRIGGERS)
    return dict(genre=genre, setting=setting, tone=tone,
                pov=pov, element=element, trigger=trigger)

# Preview a few configs
for _ in range(3):
    cfg = sample_prompt_config()
    print(cfg)
    print()

## Step 4 — Story generation function

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """Kamu adalah penulis cerpen Melayu yang berpengalaman. Kamu menulis dalam Bahasa Melayu kolokial yang natural — bukan terjemahan dari bahasa Inggeris.

Gaya penulisan kamu:
- Ayat pendek dan bersemangat, diselingi ayat yang lebih panjang untuk kedalaman
- Dialog yang natural, menggunakan kau/aku dalam perbualan biasa
- Narator mungkin bertutur terus kepada pembaca dengan nada santai
- Bina ketegangan melalui deria fizikal — bulu roma berdiri, darah hilang dari muka, menggigil
- Elemen luar biasa TIDAK pernah dijelaskan sepenuhnya — biarkan ia kekal samar
- Latar institusi Malaysia yang dikenali ramai (kilang, asrama, kampung, pejabat, hospital)
- Unsur Islam sebagai sebahagian kehidupan biasa — malam Jumaat, jam 3 pagi, Yaasin, ustaz
- Subvert jangkaan genre — mulakan sebagai satu jenis cerita, selesaikan sebagai jenis lain

Tulis HANYA teks cerpen. Tiada ulasan, tiada nota pengarang, tiada penjelasan."""


def build_story_prompt(cfg):
    return f"""Tulis sebuah cerpen Bahasa Melayu yang lengkap dengan spesifikasi berikut:

Genre: {cfg['genre']}
Latar: {cfg['setting']}
Nada: {cfg['tone']}
Sudut pandang: {cfg['pov']}
Elemen utama: {cfg['element']}
Pencetus: {cfg['trigger']}

Keperluan:
- Panjang: 600 hingga 1,000 patah perkataan
- Mulakan dengan tajuk dalam format: **Tajuk Cerita**
- Gunakan nama watak Malaysia yang realistik
- Akhiri dengan cara yang berkesan — boleh terbuka, mengejut, atau suram
- Jangan jelaskan atau rumuskan elemen supranatural secara eksplisit
- Tulis dalam Bahasa Melayu yang kolokial dan natural"""


def generate_story(cfg, max_tokens=1200):
    """Generate a single story given a config dict. Returns (title, text) or raises."""
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {'role': 'system',  'content': SYSTEM_PROMPT},
            {'role': 'user',    'content': build_story_prompt(cfg)},
        ],
        max_tokens=max_tokens,
        temperature=0.92,
        top_p=0.95,
    )
    raw = response.choices[0].message.content.strip()

    # Extract title from first line (**Title**)
    lines = raw.split('\n')
    title = lines[0].strip('*# ').strip()
    body  = '\n'.join(lines[1:]).strip()

    usage = response.usage
    return title, body, raw, usage.total_tokens


# Quick test
cfg = sample_prompt_config()
print('Config:', cfg)
title, body, raw, tokens = generate_story(cfg)
print(f'\nTitle:  {title}')
print(f'Tokens: {tokens}')
print(f'\n{raw[:500]}...')

## Step 5 — LLM judge function

Scores each story on 5 dimensions. Stories below `QUALITY_THRESHOLD` (average score) are discarded.

In [ ]:
import json

JUDGE_SYSTEM = """Kamu adalah pengkritik sastera Bahasa Melayu yang objektif. Tugasmu adalah menilai kualiti cerpen yang diberikan.

Berikan skor untuk setiap dimensi dari 1 hingga 10. Jawab HANYA dalam format JSON yang ditetapkan. Tiada ulasan tambahan."""

JUDGE_TEMPLATE = """Nilai cerpen berikut:

---
{story}
---

Berikan skor 1-10 untuk setiap dimensi:

1. bahasa — Kualiti Bahasa Melayu: tatabahasa, kosa kata, kealamian, bukan terjemahan
2. naratif — Struktur Naratif: pembukaan, konflik, perkembangan, resolusi/akhir yang berkesan
3. keaslian — Keaslian: idea segar, bukan klise, tidak generik
4. melayu — Rasa Melayu: latar, watak, dan suasana yang autentik Malaysia/Melayu
5. tarikan — Daya Tarikan: adakah pembaca akan terus membaca? Adakah ia menarik?

Format JSON yang diperlukan (angka sahaja, tiada teks):
{{
  "bahasa": <1-10>,
  "naratif": <1-10>,
  "keaslian": <1-10>,
  "melayu": <1-10>,
  "tarikan": <1-10>,
  "komen": "<satu ayat sebab skor ini>"
}}"""


def judge_story(story_text, max_tokens=200):
    """Score a story. Returns (scores_dict, average, tokens_used)."""
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {'role': 'system', 'content': JUDGE_SYSTEM},
            {'role': 'user',   'content': JUDGE_TEMPLATE.format(story=story_text[:3000])},
        ],
        max_tokens=max_tokens,
        temperature=0.1,   # low temp for consistent scoring
        response_format={'type': 'json_object'},
    )
    raw_json = response.choices[0].message.content.strip()
    scores   = json.loads(raw_json)

    dims    = ['bahasa', 'naratif', 'keaslian', 'melayu', 'tarikan']
    average = sum(scores[d] for d in dims) / len(dims)

    return scores, round(average, 2), response.usage.total_tokens


# Test judge on the story generated above
scores, avg, jtokens = judge_story(raw)
print(f'Scores:  {scores}')
print(f'Average: {avg}/10')
print(f'Tokens:  {jtokens}')
print(f'Keep?    {"YES" if avg >= QUALITY_THRESHOLD else "NO"}')

## Step 6 — HuggingFace dataset setup

In [ ]:
from huggingface_hub import HfApi, login
from datasets import Dataset, load_dataset
import pandas as pd

login(token=HF_TOKEN)
api = HfApi()

# Create the dataset repo if it doesn't exist
try:
    api.create_repo(HF_REPO_ID, repo_type='dataset', private=False)
    print(f'Created: {HF_REPO_ID}')
except Exception:
    print(f'Repo exists: {HF_REPO_ID}')

# Try to load existing data so we can append rather than overwrite
try:
    existing_ds = load_dataset(HF_REPO_ID, split='train')
    existing_records = existing_ds.to_list()
    print(f'Existing stories: {len(existing_records)}')
except Exception:
    existing_records = []
    print('No existing data — starting fresh.')

## Step 7 — Checkpoint utilities

In [ ]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'accepted': [], 'rejected': 0, 'total_tokens': 0, 'attempts': 0}


def save_checkpoint(state):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def push_to_hf(records):
    """Push all accepted records (existing + new) to HuggingFace."""
    all_records = existing_records + records
    ds = Dataset.from_list(all_records)
    ds.push_to_hub(HF_REPO_ID, split='train', token=HF_TOKEN)
    print(f'  Pushed {len(all_records)} total stories to {HF_REPO_ID}')


state = load_checkpoint()
print(f'Checkpoint loaded:')
print(f'  Accepted so far: {len(state["accepted"])}')
print(f'  Rejected so far: {state["rejected"]}')
print(f'  Attempts so far: {state["attempts"]}')
print(f'  Tokens used:     {state["total_tokens"]:,}')

## Step 8 — Main generation loop

In [ ]:
import time
from datetime import datetime

state     = load_checkpoint()
accepted  = state['accepted']     # list of story dicts already accepted
needed    = TARGET_STORIES - len(accepted)

print(f'Need {needed} more stories to reach target of {TARGET_STORIES}.')
print(f'Running...\n')

last_push_count = 0  # track when to push

while len(accepted) < TARGET_STORIES:
    state['attempts'] += 1
    attempt = state['attempts']

    cfg = sample_prompt_config()

    # ── Generate ────────────────────────────────────────────────────
    try:
        title, body, raw_text, gen_tokens = generate_story(cfg)
    except Exception as e:
        print(f'  [{attempt}] Generation error: {e}')
        time.sleep(5)
        continue

    # ── Judge ───────────────────────────────────────────────────────
    try:
        scores, avg_score, judge_tokens = judge_story(raw_text)
    except Exception as e:
        print(f'  [{attempt}] Judge error: {e}')
        time.sleep(5)
        continue

    total_tokens = gen_tokens + judge_tokens
    state['total_tokens'] += total_tokens

    # ── Filter ──────────────────────────────────────────────────────
    if avg_score < QUALITY_THRESHOLD:
        state['rejected'] += 1
        print(f'  [{attempt}] REJECTED  score={avg_score:.1f}  genre={cfg["genre"][:20]}'
              f'  tokens={total_tokens}  comment={scores.get("komen", "")[:50]}')
        save_checkpoint(state)
        time.sleep(0.5)
        continue

    # ── Accept ──────────────────────────────────────────────────────
    record = {
        'text':          f'<|story|>\n{raw_text}\n<|end|>',
        'title':         title,
        'genre':         cfg['genre'],
        'setting':       cfg['setting'],
        'tone':          cfg['tone'],
        'score':         avg_score,
        'score_bahasa':  scores.get('bahasa',  0),
        'score_naratif': scores.get('naratif', 0),
        'score_keaslian':scores.get('keaslian',0),
        'score_melayu':  scores.get('melayu',  0),
        'score_tarikan': scores.get('tarikan', 0),
        'judge_comment': scores.get('komen',   ''),
        'generated_at':  datetime.utcnow().isoformat(),
    }
    accepted.append(record)
    state['accepted'] = accepted

    print(f'  [{attempt}] ACCEPTED  score={avg_score:.1f}  '
          f'accepted={len(accepted)}/{TARGET_STORIES}  '
          f'genre={cfg["genre"][:20]}  tokens={state["total_tokens"]:,}')

    # ── Save checkpoint ─────────────────────────────────────────────
    save_checkpoint(state)

    # ── Push to HF every N accepted stories ─────────────────────────
    if len(accepted) - last_push_count >= PUSH_EVERY:
        print(f'  Pushing to HuggingFace ({len(accepted)} accepted)...')
        push_to_hf(accepted)
        last_push_count = len(accepted)

    time.sleep(0.3)   # small pause to avoid rate limits

# ── Final push ───────────────────────────────────────────────────────
print(f'\nGeneration complete!')
print(f'  Accepted:       {len(accepted)}')
print(f'  Rejected:       {state["rejected"]}')
print(f'  Total attempts: {state["attempts"]}')
print(f'  Total tokens:   {state["total_tokens"]:,}')
print(f'  Accept rate:    {len(accepted)/state["attempts"]*100:.1f}%')
print()
print('Pushing final batch to HuggingFace...')
push_to_hf(accepted)

## Step 9 — Inspect results

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(HF_REPO_ID, split='train')
df = ds.to_pandas()

print(f'Total stories in dataset: {len(df)}')
print(f'\nScore distribution:')
print(df['score'].describe().round(2))

print(f'\nStories per genre:')
print(df['genre'].value_counts().to_string())

print(f'\nAverage score per genre:')
print(df.groupby('genre')['score'].mean().sort_values(ascending=False).round(2).to_string())

In [ ]:
# Read a random accepted story
sample = df.sample(1).iloc[0]
print(f'Title:   {sample["title"]}')
print(f'Genre:   {sample["genre"]}')
print(f'Score:   {sample["score"]}/10')
print(f'Comment: {sample["judge_comment"]}')
print(f'Score breakdown: bahasa={sample["score_bahasa"]} naratif={sample["score_naratif"]} '
      f'keaslian={sample["score_keaslian"]} melayu={sample["score_melayu"]} tarikan={sample["score_tarikan"]}')
print()
print(sample['text'])

In [ ]:
# Show the top 5 highest-scoring stories
print('Top 5 stories by score:\n')
for _, row in df.nlargest(5, 'score').iterrows():
    print(f'  {row["score"]:.1f}  [{row["genre"]}]  {row["title"]}')
    print(f'        {row["judge_comment"]}')

## Step 10 — Token usage summary

Track daily token consumption against the 10M free limit.

In [ ]:
state = load_checkpoint()

total_tokens     = state['total_tokens']
accepted_count   = len(state['accepted'])
rejected_count   = state['rejected']
attempt_count    = state['attempts']

tokens_per_story = total_tokens / attempt_count if attempt_count else 0
remaining_budget = 10_000_000 - total_tokens
stories_left     = int(remaining_budget / tokens_per_story) if tokens_per_story else 0

print('═' * 50)
print('  Token Usage Summary')
print('═' * 50)
print(f'  Total tokens used:    {total_tokens:>10,}')
print(f'  Daily limit:          {10_000_000:>10,}')
print(f'  Remaining today:      {remaining_budget:>10,}  ({remaining_budget/10_000_000*100:.1f}%)')
print(f'  Tokens per attempt:   {tokens_per_story:>10.0f}')
print(f'  Estimated stories:    {stories_left:>10,}  remaining today')
print('─' * 50)
print(f'  Accepted:             {accepted_count:>10,}')
print(f'  Rejected:             {rejected_count:>10,}')
print(f'  Accept rate:          {accepted_count/attempt_count*100:>9.1f}%' if attempt_count else '')
print('═' * 50)